<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M11/M11_Lab1_Vision_Multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">👁️ M11 Lab 1 — Vision & Multimodal AI</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Send <b>images to GPT-4V</b> via the API (URL and base64 encoding)</li>
    <li>Perform vision tasks: <b>describe, OCR, chart analysis, image comparison</b></li>
    <li>Compare <b>GPT-4V vs Gemini Vision</b> on the same image</li>
    <li>Understand <b>multimodal prompting</b> techniques</li>
    <li>Build a practical <b>image analysis pipeline</b></li>
  </ol>
</div>

## 📦 Setup

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.
>
> For the Gemini comparison: also add `GEMINI_API_KEY`.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai google-generativeai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, pp, show_response, show_expected, show_info, compare_responses, quiz

client = setup_openai()

import base64
import requests
from IPython.display import Image, display

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Sending Images via URL</h2>
</div>

The simplest way to send an image to GPT-4V is via a **public URL**. The API downloads the image and analyzes it.

In [ ]:
# ============================================================
# 🖼️ Task 1: Describe an Image (URL)
# ============================================================

# Public domain image: a supply chain warehouse
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Walmart_store_exterior_5266815680.jpg/1280px-Walmart_store_exterior_5266815680.jpg"

# Show the image
display(Image(url=image_url, width=500))

# Send to GPT-4V
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in detail. What do you see? What business or industry does this relate to?"},
            {"type": "image_url", "image_url": {"url": image_url}}
        ]
    }],
    max_tokens=500
)

print(f"🤖 GPT-4V Description:\n{response.choices[0].message.content}")
print(f"\n📊 Tokens used: {response.usage.total_tokens}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Vision API calls use more tokens than text-only calls. An image typically costs 85-1,105 tokens depending on size and detail level. Use <code>detail: "low"</code> for cheaper analysis or <code>detail: "high"</code> for fine-grained tasks like OCR.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Sending Images via Base64</h2>
</div>

For local images or when you don't have a public URL, encode the image as **base64** and embed it directly in the API call.

In [ ]:
# ============================================================
# 🖼️ Task 2: Send an Image via Base64
# ============================================================

def analyze_image_url(image_url: str, question: str) -> str:
    """Download an image, encode as base64, and send to GPT-4V."""
    # Download the image
    img_data = requests.get(image_url).content
    b64_image = base64.b64encode(img_data).decode("utf-8")

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {
                    "url": f"data:image/jpeg;base64,{b64_image}",
                    "detail": "high"
                }}
            ]
        }],
        max_tokens=500
    )
    return response.choices[0].message.content

# OCR Task — read text from an image
# Using a public domain image with text
text_image = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/2f/Google_2015_logo.svg/1200px-Google_2015_logo.svg.png"

display(Image(url=text_image, width=400))
result = analyze_image_url(text_image, "What text do you see in this image? Read it exactly as shown.")
print(f"\n📝 OCR Result: {result}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Chart & Data Analysis</h2>
</div>

One of the most practical uses of vision AI: **analyzing charts and graphs**. Let's create a chart and have GPT-4V interpret it.

In [ ]:
# ============================================================
# 📊 Task 3: Create a Chart and Have AI Analyze It
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import io

# Create a sample supply chain chart
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#1a1a2e')

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
demand = [100, 110, 95, 130, 145, 160, 155, 170, 180, 165, 190, 210]
supply = [120, 105, 110, 115, 140, 150, 165, 160, 175, 170, 185, 200]

ax.plot(months, demand, 'o-', color='#ff6b6b', linewidth=2, label='Demand', markersize=6)
ax.plot(months, supply, 's-', color='#4ecdc4', linewidth=2, label='Supply', markersize=6)
ax.fill_between(range(12), demand, supply, alpha=0.1, color='#ff6b6b')

ax.set_title('Supply vs Demand — 2024', color='white', fontsize=16, fontweight='bold')
ax.set_ylabel('Units (thousands)', color='white', fontsize=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1a2e', edgecolor='#333', labelcolor='white')
ax.grid(alpha=0.2, color='white')

plt.tight_layout()

# Save to buffer and encode
buf = io.BytesIO()
plt.savefig(buf, format='png', dpi=150, facecolor=fig.get_facecolor())
buf.seek(0)
chart_b64 = base64.b64encode(buf.read()).decode('utf-8')
plt.show()

# Send chart to GPT-4V
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": (
                "Analyze this supply chain chart. Provide:\n"
                "1. What trend do you see in demand vs supply?\n"
                "2. In which months is there a supply shortage?\n"
                "3. What business recommendation would you make?"
            )},
            {"type": "image_url", "image_url": {
                "url": f"data:image/png;base64,{chart_b64}"
            }}
        ]
    }],
    max_tokens=500
)

print(f"📊 Chart Analysis:\n{response.choices[0].message.content}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ GPT-4V vs Gemini Vision</h2>
</div>

Let's compare how two different vision models analyze the same image.

In [ ]:
# ============================================================
# 🔄 Compare GPT-4V vs Gemini Vision
# ============================================================
import google.generativeai as genai
from google.colab import userdata
from PIL import Image as PILImage

# Configure Gemini
try:
    genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
    gemini_available = True
    print("✅ Gemini API configured")
except Exception:
    gemini_available = False
    print("⚠️ GEMINI_API_KEY not found — skipping Gemini comparison")

# Download a test image
test_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/1280px-Camponotus_flavomarginatus_ant.jpg"
img_data = requests.get(test_url).content
b64_img = base64.b64encode(img_data).decode("utf-8")

display(Image(url=test_url, width=400))

question = "Identify what's in this image. What species is it? Describe 3 interesting facts about it."

# GPT-4V
gpt_resp = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": [
        {"type": "text", "text": question},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}}
    ]}],
    max_tokens=400
)
gpt_text = gpt_resp.choices[0].message.content

# Gemini Vision
if gemini_available:
    import io as _io
    pil_img = PILImage.open(_io.BytesIO(img_data))
    gemini_model = genai.GenerativeModel("gemini-2.5-flash")
    gemini_resp = gemini_model.generate_content([question, pil_img])
    gemini_text = gemini_resp.text

    compare_responses({
        "GPT-4V": gpt_text,
        "Gemini Vision": gemini_text,
    })
else:
    show_response(gpt_resp)
    print("\n⚠️ Add GEMINI_API_KEY to Colab Secrets to compare with Gemini Vision.")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Different vision models have different strengths. GPT-4V tends to be more detailed in descriptions, while Gemini Vision often provides more structured output. In production, you might use <b>both models</b> and compare results for critical tasks.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What are the two ways to send an image to GPT-4V via the API?",
        "options": [
            "Upload to S3 first, then send the bucket name",
            "Public URL or base64-encoded data embedded in the request",
            "Send the file path on your local computer",
            "Convert to text description first, then send the text"
        ],
        "answer": 1,
        "explanation": "You can either provide a public URL (the API downloads it) or encode the image as base64 and embed it in the request as a data URI. Base64 is useful when images aren't publicly accessible."
    },
    {
        "q": "Why do vision API calls cost more tokens than text-only calls?",
        "options": [
            "Images are processed by a separate, more expensive model",
            "The image is tokenized into visual tokens (85-1,105 depending on size/detail), adding to the input token count",
            "Vision calls require a premium API key",
            "The API charges extra for the bandwidth to download images"
        ],
        "answer": 1,
        "explanation": "Images are broken into visual tokens — similar to how text is broken into text tokens. A low-detail image costs ~85 tokens; a high-detail image can cost over 1,000 tokens. This is why the 'detail' parameter matters for cost control."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Upload Your Own Image

Upload an image from your computer and get AI analysis. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: Analyze Your Own Image
# ============================================================
# Replace each ----- with the correct value

from google.colab import files
import io as _io

# Upload an image
print("📁 Upload an image file:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    img_bytes = uploaded[filename]

    # Display the image
    display(Image(data=img_bytes, width=400))

    # Encode as base64
    b64 = base64.------(img_bytes).decode("utf-8")   # What encoding function?

    # Ask AI to analyze
    response = client.chat.completions.create(
        model="-----",                               # Which model supports vision?
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Analyze this image. Describe what you see, identify key elements, and suggest how this image could be used in a business context."},
                {"type": "-----", "image_url": {      # What content type for images?
                    "url": f"data:image/png;base64,{b64}"
                }}
            ]
        }],
        max_tokens=-----                              # How many tokens for a detailed response?
    )
    print(f"\n🤖 Analysis:\n{response.choices[0].message.content}")
else:
    print("No file uploaded.")

**📝 Your Observations:** *(double-click to edit)*

1. Which model described the image more accurately? More creatively? _[Your answer]_

2. How accurate was the AI's description of your uploaded image? _[Your answer]_

3. What types of images are vision AI best and worst at analyzing? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these vision tasks on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Receipt OCR:</b> Take a photo of a receipt and extract all items and prices</li>
    <li><b>Diagram understanding:</b> Send a flowchart or architecture diagram and ask AI to explain it</li>
    <li><b>Image comparison:</b> Send two similar images and ask the model to describe the differences</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You mastered multimodal AI — sending images via URL and base64, OCR, chart analysis, and model comparison.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li>Images can be sent via <b>URL</b> or <b>base64</b> encoding</li>
      <li><code style="color: #90caf9;">detail: "high"</code> for OCR, <code style="color: #90caf9;">detail: "low"</code> for cheap analysis</li>
      <li>Vision AI excels at <b>description, OCR, chart reading, comparison</b></li>
      <li>Always <b>compare models</b> — GPT-4V and Gemini have different strengths</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M11 Lab 2: LLM Evaluation Framework</p>
</div>